# CSI Training on Kaggle - Stable Workflow
Notebook nay theo quy trinh on dinh lau dai: clone tag co dinh tu GitHub → train → log metadata → output

Chi tiet: xem KAGGLE_WORKFLOW.md trong repo

## 1) Prerequisites
- Settings: bat GPU Accelerator + Internet
- Add Data: add Kaggle Dataset chua data (neu data k trong repo)
- GitHub: repo public, da co tag (vi du: v1.0)

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('torch') is None:
    print('PyTorch chua co, dang cai...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchvision', 'torchaudio'])

import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

assert torch.cuda.is_available(), 'GPU not available. Go to Settings > Accelerator > GPU'

PyTorch chua co, dang cai...
PyTorch: 2.10.0+cpu
CUDA available: False


AssertionError: Chua bat GPU runtime trong Colab. Vao Runtime > Change runtime type > GPU.

## 2) Clone Repository & Checkout Tag
Clone tu GitHub public repo, checkout tag co dinh (snapshot stable)

In [ ]:
import subprocess
import os

# SO: https://github.com/<your-username>/<your-repo>
REPO_URL = 'https://github.com/your-username/PBL5-train-model.git'
PROJECT_DIR = '/kaggle/working/PBL5-train-model'
TAG = 'v1.0'  # DOI TAG NAY DUNG VOI VERSION BAN MUON TRAIN

# Clone main branch
!git clone --branch main {REPO_URL} {PROJECT_DIR}
os.chdir(PROJECT_DIR)

# Checkout tag co dinh
!git checkout {TAG}

# Kiem tra
commit_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()
print('=' * 60)
print(f'Tag: {TAG}')
print(f'Commit SHA: {commit_sha}')
print(f'Working dir: {os.getcwd()}')
print('=' * 60)

## 3) Copy Data & Install Dependencies

In [ ]:
import shutil
import os

# Copy data tu Kaggle Dataset
# KHI LAM: Dang Kaggle Dataset input folder tren Settings
DATA_INPUT = '/kaggle/input/pbl5-v1-data'
if os.path.isdir(DATA_INPUT):
    shutil.copy(f'{DATA_INPUT}/sit.csv', 'data/raw/sit.csv')
    shutil.copy(f'{DATA_INPUT}/stand.csv', 'data/raw/stand.csv')
    print('Data copied from Kaggle Dataset')
else:
    print(f'Data folder {DATA_INPUT} not found, checking local data...')

# Kiem tra data
required_files = [
    'data/raw/sit.csv',
    'data/raw/stand.csv',
    'configs/train_default.json',
]
missing = [p for p in required_files if not os.path.exists(p)]
assert not missing, f'Missing: {missing}'

# Cai package
!pip install -q -r requirements.txt

print('Dependencies & data OK')

## 4) Train Model + Log Metadata
Train va automatically ghi lai metadata (tag, commit, date, config)

In [ ]:
import json
import subprocess
from datetime import datetime
import os

# Lay commit SHA
commit_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()
tag = subprocess.check_output(['git', 'describe', '--tags']).decode().strip()

# Metadata run
metadata = {
    'date': datetime.now().isoformat(),
    'tag': tag,
    'commit_sha': commit_sha,
    'model_type': 'cnn2d',
    'epochs': 20,
    'batch_size': 16,
    'use_hampel': True,
}

print('Training CNN2D model...')
!python -m src.train \
  --config configs/train_default.json \
  --model-type cnn2d \
  --epochs 20 \
  --batch-size 16 \
  --run-name "kaggle_{tag}_cnn2d"

# Luu metadata
results_dir = f'experiments/results/kaggle_{tag}_cnn2d'
metadata_file = f'{results_dir}/metadata.json'
with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Metadata saved: {metadata_file}')

## 5) (Optional) Train LSTM-CNN Variant

In [ ]:
import json
import subprocess
from datetime import datetime

# Lay tag va commit
commit_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()
tag = subprocess.check_output(['git', 'describe', '--tags']).decode().strip()

# Metadata
metadata = {
    'date': datetime.now().isoformat(),
    'tag': tag,
    'commit_sha': commit_sha,
    'model_type': 'lstmcnn',
    'epochs': 30,
    'batch_size': 16,
}

print('Training LSTM-CNN model...')
!python -m src.train \
  --config configs/train_default.json \
  --model-type lstmcnn \
  --epochs 30 \
  --batch-size 16 \
  --run-name "kaggle_{tag}_lstmcnn"

# Luu metadata
results_dir = f'experiments/results/kaggle_{tag}_lstmcnn'
metadata_file = f'{results_dir}/metadata.json'
with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Metadata saved: {metadata_file}')

## 6) Check Results & Checkpoints

In [ ]:
import os

print('Checkpoints:')
!ls -lah models/checkpoints/

print('\nResults folders:')
!ls -lah experiments/results/

print('\nEach result folder has:')
result_folders = os.listdir('experiments/results')[:1]
if result_folders:
    sample_folder = f'experiments/results/{result_folders[0]}'
    print(f'  Contents of {result_folders[0]}:')
    !ls -lah {sample_folder}

## 7) Pack Artifacts for Download
Dong goi checkpoint & results thanh zip trong /kaggle/working ro tai ve

In [ ]:
import os
import shutil
import subprocess

# Get tag
tag = subprocess.check_output(['git', 'describe', '--tags']).decode().strip()

os.makedirs('/kaggle/working/output', exist_ok=True)

# Nén checkpoint
print('Packing checkpoints...')
shutil.make_archive(
    f'/kaggle/working/output/checkpoints_{tag}',
    'zip',
    'models/checkpoints'
)

# Nén results
print('Packing results...')
shutil.make_archive(
    f'/kaggle/working/output/results_{tag}',
    'zip',
    'experiments/results'
)

print('\n' + '=' * 60)
print('ARTIFACTS READY FOR DOWNLOAD:')
print('=' * 60)
for f in os.listdir('/kaggle/working/output'):
    size = os.path.getsize(f'/kaggle/working/output/{f}') / (1024 * 1024)  # MB
    print(f'{f}: {size:.2f} MB')
print('\nDownload from Notebook Output tab')
print('=' * 60)